# HumanAI Detect - Ozellik Cikarimi + Held-out Yenileme (Asama 3+4, Colab GPU)

**Baglam:** Ana `human`/`ai_raw`/`ai_humanized` havuzlarina GPT-4o-mini ile uretilen
cok-ureticili takviye (1000 ai_raw_openai + 999 ai_humanized_openai) birlestirildi
(bkz. proje notlari, 2026-07-19). Yeni toplam: human=3262, ai_raw=4110, ai_humanized=4106
(**11478 ornek**, onceki 9479'dan +1999).

**Neden Colab:** Yerelde `build_features.py` calistirildi ama **iki gercek bug** bulundu:
1. Embedding cache'i TUM metin listesini tek hash altinda tutuyordu -- tek yeni ornek
   eklenince onbellek tamamen gecersiz oluyordu.
2. Cache path'i cift-katlanmis bir klasore yaziyordu, hic kullanilmiyordu.

Ikisi de duzeltildi (commit `af023cf`, push edildi) -- embedding cache artik ORNEK
BAZINDA calisiyor. Ama **bu, ilk (bu) calistirmada hala TUM 11478 ornegin embedding'inin
sifirdan hesaplanmasi gerektigi anlamina geliyor** (hicbir gecerli onbellek yok, hicbir
zaman olmadi). Yerel CPU'da bu ~5 saat surdugu icin (olculdu, sonra kesildi) Colab GPU'ya
tasindi.

**Iki-gecisli akis (onemli, neden asagida acikliyor):** `build_features.py` scaler/uzunluk-
residualizasyonunu sizintisiz yapabilmek icin `holdout_ids.txt`'e ihtiyac duyuyor, ama
`holdout_ids.txt` da `groups.parquet`'ten (build_features.py'nin kendi ciktisi) turetiliyor
-- yumurta-tavuk sorunu. Cozum (bu projede daha once de kullanilan yontem):
1. **1. gecis:** `build_features.py`'yi holdout OLMADAN calistir (gecici, tum veriyle fit).
2. `make_holdout_split.py` ile YENI (11478 orneği icine alan) `holdout_ids.txt`'i uret.
3. **2. gecis:** `build_features.py`'yi TEKRAR calistir -- bu sefer holdout_ids.txt mevcut,
   scaler/residualizer sadece train'den fit edilir (sizintisiz). **Bu ikinci gecis COK
   hizli olacak** cunku embedding'ler artik onbellekte (1. gecisten kalma), sadece
   stilometri+fusion adimi tekrarlanacak.

**Onemli -- neden holdout'u yeniden uretmek gerekiyor:** Eski `holdout_ids.txt` GPT-4o-mini
orneklerinden hicbirini icermiyordu (onceden var olmadiklari icin) -- yani resmi held-out
degerlendirmesi capraz-uretici genellemeyi hic yansitmayacakti. Yeni holdout, tum
ureticilerden orantili orneklenmis olacak.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')

In [ ]:
# 2. Repo klonla (guncel kod, commit af023cf+ -- embedding cache duzeltmesi bu commit'te)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -5

In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q

In [ ]:
# 4. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 5. Guncel (birlestirilmis) interim dosyalarini yerel makineden yukle:
#    human.jsonl (3262), ai_raw.jsonl (4110), ai_humanized.jsonl (4106) -- UCUNU BIRDEN sec
# Bu dosyalar buyuk (~240-350MB her biri, toplam ~900MB) -- yuklemesi birkac dakika surebilir.
from pathlib import Path
from google.colab import files

for label in ['human', 'ai_raw', 'ai_humanized']:
    Path(f'data/interim/{label}').mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized' in name:
        Path(name).rename('data/interim/ai_humanized/ai_humanized.jsonl')
    elif 'ai_raw' in name:
        Path(name).rename('data/interim/ai_raw/ai_raw.jsonl')
    elif 'human' in name:
        Path(name).rename('data/interim/human/human.jsonl')

for label in ['human', 'ai_raw', 'ai_humanized']:
    p = Path(f'data/interim/{label}/{label}.jsonl')
    n = sum(1 for _ in open(p, encoding='utf-8')) if p.exists() else 0
    print(f'{label}: {n} ornek')

In [ ]:
# 6. 1. GECIS: build_features.py -- holdout_ids.txt YOK, tum veriyle gecici fit.
# Bu adim UZUN surer (11478 ornegin TAMAMININ BERTurk embedding'i sifirdan hesaplaniyor,
# hicbir gecerli onbellek olmadigi icin). GPU'da tahmini ~30-60 dk (CPU'daki ~5 saate kiyasla).
!python scripts/build_features.py

In [ ]:
# 7. YENI holdout_ids.txt uret (artik ai_raw_openai/ai_humanized_openai orneklerini de iceriyor)
!python scripts/make_holdout_split.py
!wc -l data/processed/holdout_ids.txt

In [ ]:
# 8. 2. GECIS: build_features.py'yi TEKRAR calistir -- artik holdout_ids.txt var, scaler/
# uzunluk-residualizasyonu SADECE train'den (sizintisiz) fit edilecek. Embedding'ler onbellekte
# oldugu icin bu gecis COK HIZLI olmali (sadece stilometri+fusion, birkac dakika).
#
# ONEMLI: build_features.py, stylometric/embeddings/fused ciktilarindan herhangi biri zaten
# VARSA atlar -- bu yuzden 2. gecisten once bunlari silmemiz, ama embedding CACHE'ini
# (data/processed/embedding_cache/) SILMEMEMIZ lazim (asil hiz kazanci ondan geliyor).
from pathlib import Path
for f in ['stylometric.parquet', 'fused.parquet', 'groups.parquet', 'length_residualizer.json',
          'embeddings_berturk.parquet']:
    p = Path('data/processed') / f
    if p.exists():
        p.unlink()
        print(f'silindi: {p}')

!python scripts/build_features.py

In [ ]:
# 9. Sonuclari yerel makineye indir
from pathlib import Path
from google.colab import files

for f in ['stylometric.parquet', 'embeddings_berturk.parquet', 'groups.parquet',
          'fused.parquet', 'holdout_ids.txt', 'length_residualizer.json']:
    src = Path('data/processed') / f
    if src.exists():
        print(f'indiriliyor: {f}')
        files.download(str(src))
    else:
        print(f'UYARI: {f} bulunamadi.')

## Yerel Makineye Aktarim

Yukaridaki hucre 6 dosyayi indirir: `stylometric.parquet`, `embeddings_berturk.parquet`,
`groups.parquet`, `fused.parquet`, `holdout_ids.txt`, `length_residualizer.json`.

Kendi makinende hepsini `data/processed/` altina (mevcutlarin UZERINE) koy, sonra bana
haber ver -- sonraki adim: `colab_measure_calibration.ipynb` ile (mevcut notebook, degisiklik
gerekmiyor) modeli yeniden egitip kalibre edecegim. **Not:** o notebook'ta "GPU runtime"
kullanma -- egitim adimi (sklearn/XGBoost/CatBoost) GPU kullanmiyor, saf CPU-bound; "CPU
+ Yuksek RAM" runtime tercih edilmeli (bkz. proje notlari, GPU runtime bu is icin daha az
vCPU verip yavaslatiyordu).